In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.metrics import confusion_matrix, roc_curve, auc
from sklearn.impute import SimpleImputer

Columns_to_Drop = ['Row Id', 'Row Id.', 'TARGET_D']

df_fundraising = pd.read_csv("sample_data/Fundraising.csv")
df_future_fundraising = pd.read_csv("sample_data/FutureFundraising.csv")
df_fundraising.head()
df_fundraising.info()
df_future_fundraising.head()
df_future_fundraising.info()

df_fundraising_cleaned = df_fundraising.drop(Columns_to_Drop, axis=1)
df_future_fundraising_cleaned = df_future_fundraising.drop(Columns_to_Drop, axis=1)
df_fundraising_cleaned.rename(columns={'gender dummy': 'gender_dummy', 'homeowner dummy': 'homeowner_dummy'}, inplace=True)
df_future_fundraising_cleaned.rename(columns={'gender dummy': 'gender_dummy', 'homeowner dummy': 'homeowner_dummy'}, inplace=True)
X = df_fundraising_cleaned.drop(('TARGET_B'), axis=1)
Y = df_fundraising_cleaned['TARGET_B']
X_train, X_validation, Y_train, Y_validation = train_test_split(X, Y, test_size=.40, random_state=12345)

numerical_cols = [col for col in X_train.columns if X_train[col].nunique() > 21 and X_train[col].dtype != 'object']

pipeline_log_reg = Pipeline([
    ('scaler', StandardScaler()),
    ('lr_model', LogisticRegression(
        penalty='l2',
        C = 1.0,
        random_state = 12345
    ))
])

pipeline_svc = Pipeline([
    ('scaler', StandardScaler()),
    ('svc_model', SVC(
        kernel = 'rbf',
        C = 1.0,
        random_state = 12345,
        probability = True
    ))
])

model_trained_log_reg = pipeline_log_reg.fit(X_train, Y_train)
model_trained_svc = pipeline_svc.fit(X_train, Y_train)

Y_validation_prob_log_reg = model_trained_log_reg.predict_proba(X_validation) [:, 1]
Y_validation_prob_svc = model_trained_svc.predict_proba(X_validation) [:, 1]

validation_data_results = X_validation.copy()
validation_data_results['TARGET_B'] = Y_validation.values
validation_data_results['Y_validation_prob_log_reg'] = Y_validation_prob_log_reg
validation_data_results['Y_validation_prob_svc'] = Y_validation_prob_svc

Cost_per_mail = 0.68
Expected_donation = 13.00
Weight_donor = 0.50/0.051
Weight_non_donor = 0.50/0.949

def calculate_adjusted_net_profits(data_row):
  if data_row['TARGET_B'] == 1:
    net_profit_unadjusted = Expected_donation - Cost_per_mail
    weight = Weight_donor
  else:
    net_profit_unadjusted = 0.00 - Cost_per_mail
    weight = Weight_non_donor
  adjusted_net_profit = net_profit_unadjusted/weight
  return adjusted_net_profit

validation_data_results['Adjusted_Profit'] = validation_data_results.apply(calculate_adjusted_net_profits, axis = 1)

def calculate_cumulative_gains(data, probability_column, profit_column):
  sorted_data = data.sort_values(by=probability_column, ascending=False).reset_index(drop = True)
  sorted_data['Cumulative_Net_Profit'] = sorted_data[profit_column].cumsum()
  N = len(sorted_data)
  sorted_data['Proportion_Mailed'] = (sorted_data.index + 1) / N
  return sorted_data[['Proportion_Mailed', 'Cumulative_Net_Profit']]

cumulative_gains_log_reg = calculate_cumulative_gains(validation_data_results, 'Y_validation_prob_log_reg', 'Adjusted_Profit')
cumulative_gains_svc = calculate_cumulative_gains(validation_data_results, 'Y_validation_prob_svc', 'Adjusted_Profit')

plt.figure(figsize=(10, 6))
plt.plot(cumulative_gains_log_reg['Proportion_Mailed'], cumulative_gains_log_reg['Cumulative_Net_Profit'], label='Logistic Regression')
plt.plot(cumulative_gains_svc['Proportion_Mailed'], cumulative_gains_svc['Cumulative_Net_Profit'], label='SVC')
plt.title('Cumulative Net Profit Gains on Validation Set')
plt.xlabel('Proportion of List that has been Mailed')
plt.ylabel('Cumulative Net Profit in ($)')
plt.grid(True)
plt.legend()
plt.show()

max_profit_log_reg = cumulative_gains_log_reg['Cumulative_Net_Profit'].max()
max_profit_svc = cumulative_gains_svc['Cumulative_Net_Profit'].max()

if max_profit_log_reg > max_profit_svc:
  print(f"The best model that has been determined is: Logistic Regression (Max Profit: ${max_profit_log_reg:.2f})")
  best_model = model_trained_log_reg
else:
  print(f"The best model that has been determined is: SVC (Max Profit: ${max_profit_svc:.2f})")
  best_model = model_trained_svc

Training_Features = X_train.columns
X_future = df_future_fundraising_cleaned[Training_Features]
future_probabilities = best_model.predict_proba(X_future)[:, 1]
df_future_fundraising_cleaned['Prob_Donor_Best'] = future_probabilities
final_mailing_list = df_future_fundraising_cleaned.sort_values(by='Prob_Donor_Best', ascending = False)

print("Top 10 Candidates for Mailing List:")
print(final_mailing_list[['Prob_Donor_Best']].head(10))

if max_profit_log_reg >= max_profit_svc:
    cg_best = cumulative_gains_log_reg
    model_name = 'Logistic Regression'
    max_profit = max_profit_log_reg
else:
    cg_best = cumulative_gains_svc
    model_name = 'SVC'
    max_profit = max_profit_svc

optimal_index = cg_best['Cumulative_Net_Profit'].idxmax()

optimal_proportion = cg_best.loc[optimal_index, 'Proportion_Mailed']

total_future_candidates = len(df_future_fundraising_cleaned)
cutoff_index = int(optimal_proportion * total_future_candidates)

print(f"Optimal Mailing Strategy:")
print(f"Best Model: {model_name}")
print(f"Maximum Net Profit: ${max_profit:.2f}")
print(f"Optimal Mailing Proportion: {optimal_proportion:.2%}")
print(f"Recommended number of candidates to mail: {cutoff_index} out of {total_future_candidates}")
